In [ ]:
import os
import shutil
import cv2
import numpy as np
import insightface
import json
import time
import threading
import sqlite3
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from sklearn.cluster import DBSCAN
from sklearn.ensemble import IsolationForest
from scipy.spatial.distance import cosine
from scipy import stats
from flask import Flask, render_template_string, request, redirect, url_for, flash, jsonify, Response, send_file
from PIL import Image, ImageOps, ExifTags
import io
import base64
from datetime import datetime
import pickle

# ==============================================================================
# 1. FLASK APP CONFIGURATION
# ==============================================================================
app = Flask(__name__)
app.secret_key = 'face-clustering-with-square-thumbnails'
app.config['DATABASE'] = 'face_clustering.db'
app.config['THUMBNAILS_DIR'] = 'static/image_thumbnails'
app.config['SUGGESTION_CROPS_DIR'] = 'static/suggestion_crops'

# Create directories
os.makedirs(app.config['THUMBNAILS_DIR'], exist_ok=True)
os.makedirs(app.config['SUGGESTION_CROPS_DIR'], exist_ok=True)

# --- Configuration ---
THUMBNAIL_MAX_WIDTH = 300
THUMBNAIL_MAX_HEIGHT = 300
THUMBNAIL_QUALITY = 85
DEFAULT_THREAD_COUNT = 4

# --- Progress Tracking ---
progress_data = {
    'current': 0, 'total': 0, 'text': 'Waiting...', 'faces_detected': 0,
    'images_processed': 0, 'faces_processed': 0, 'clusters_formed': 0,
    'suggestions_generated': 0, 'thumbnails_created': 0, 'start_time': None,
    'elapsed_time': 0, 'current_stage': 'idle', 'processing_speed': 0,
    'eta_seconds': 0, 'is_processing': False, 'current_image': ''
}
progress_lock = threading.Lock()
processing_thread = None
suggestions_cache = []

# --- Model Configuration ---
DEFAULT_MODEL_STATE = {
    'eps': 0.35, 'suggestion_eps': 0.60, 'total_decisions': 0,
    'accuracy_history': [], 'outliers_detected': 0,
    'created_at': time.time(), 'last_updated': time.time()
}

feedback_history = []
LEARNING_RATE = 0.15
MIN_FEEDBACK_FOR_UPDATE = 10
UPDATE_FREQUENCY = 5
EPS_MIN_BOUND = 0.1
EPS_MAX_BOUND = 0.8
ACCURACY_HISTORY_MAX = 100

# Load model state
def load_model_state():
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT value FROM app_settings WHERE key = 'model_state'")
        result = cursor.fetchone()
        conn.close()
        if result:
            return pickle.loads(result[0])
    except Exception as e:
        print(f"Error loading model state: {e}")
    return DEFAULT_MODEL_STATE.copy()

def save_model_state():
    try:
        model_state['last_updated'] = time.time()
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("INSERT OR REPLACE INTO app_settings (key, value) VALUES ('model_state', ?)", 
                      (pickle.dumps(model_state),))
        conn.commit()
        conn.close()
    except Exception as e:
        print(f"Error saving model state: {e}")

model_state = load_model_state()

# --- InsightFace Model ---
print("Initializing FaceAnalysis model...")
try:
    face_app = insightface.app.FaceAnalysis(name='buffalo_l', providers=['CUDAExecutionProvider'])
    face_app.prepare(ctx_id=0, det_size=(640, 640))
    print("InsightFace model loaded successfully on GPU.")
except Exception:
    try:
        face_app = insightface.app.FaceAnalysis(name='buffalo_l', providers=['CPUExecutionProvider'])
        face_app.prepare(ctx_id=0, det_size=(640, 640))
        print("InsightFace model loaded successfully on CPU.")
    except Exception as e:
        print(f"Fatal error: Could not load InsightFace model: {e}")
        exit()

# ==============================================================================
# 2. DATABASE FUNCTIONS
# ==============================================================================
def get_db_connection():
    conn = sqlite3.connect(app.config['DATABASE'], timeout=30.0)
    conn.execute('PRAGMA foreign_keys = ON')
    conn.execute('PRAGMA journal_mode = WAL')
    conn.row_factory = sqlite3.Row
    return conn

def init_database():
    conn = get_db_connection()
    cursor = conn.cursor()
    
    # Images table
    cursor.execute('''CREATE TABLE IF NOT EXISTS images (
        id INTEGER PRIMARY KEY AUTOINCREMENT, path TEXT UNIQUE NOT NULL, filename TEXT NOT NULL,
        content_hash TEXT UNIQUE NOT NULL, width INTEGER NOT NULL, height INTEGER NOT NULL,
        file_size INTEGER NOT NULL, file_modified_time REAL NOT NULL, has_thumbnail BOOLEAN DEFAULT FALSE,
        thumbnail_path TEXT, thumbnail_width INTEGER, thumbnail_height INTEGER, thumbnail_size INTEGER,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP, indexed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    
    # Faces table
    cursor.execute('''CREATE TABLE IF NOT EXISTS faces (
        id INTEGER PRIMARY KEY AUTOINCREMENT, image_id INTEGER NOT NULL, bbox_x1 INTEGER NOT NULL,
        bbox_y1 INTEGER NOT NULL, bbox_x2 INTEGER NOT NULL, bbox_y2 INTEGER NOT NULL,
        embedding BLOB NOT NULL, confidence REAL NOT NULL, quality_score REAL NOT NULL,
        face_area INTEGER NOT NULL, created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (image_id) REFERENCES images (id) ON DELETE CASCADE)''')
    
    # Persons table
    cursor.execute('''CREATE TABLE IF NOT EXISTS persons (
        id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT NOT NULL, representative_face_id INTEGER,
        representative_image_id INTEGER, face_count INTEGER DEFAULT 0, centroid_embedding BLOB,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP, updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (representative_face_id) REFERENCES faces (id),
        FOREIGN KEY (representative_image_id) REFERENCES images (id))''')
    
    # Person-Face associations
    cursor.execute('''CREATE TABLE IF NOT EXISTS person_faces (
        id INTEGER PRIMARY KEY AUTOINCREMENT, person_id INTEGER NOT NULL, face_id INTEGER NOT NULL,
        distance_to_centroid REAL, assigned_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        FOREIGN KEY (person_id) REFERENCES persons (id) ON DELETE CASCADE,
        FOREIGN KEY (face_id) REFERENCES faces (id) ON DELETE CASCADE, UNIQUE(person_id, face_id))''')
    
    # App settings
    cursor.execute('''CREATE TABLE IF NOT EXISTS app_settings (
        key TEXT PRIMARY KEY, value BLOB NOT NULL, updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP)''')
    
    # Indexes
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_images_hash ON images(content_hash)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_faces_image ON faces(image_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_person_faces_person ON person_faces(person_id)')
    
    conn.commit()
    conn.close()

init_database()

# ==============================================================================
# 3. IMAGE PROCESSING FUNCTIONS
# ==============================================================================
def fix_image_orientation(image_path):
    try:
        with Image.open(image_path) as img:
            img = ImageOps.exif_transpose(img)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    except Exception as e:
        print(f"Error fixing orientation for {image_path}: {e}")
        return cv2.imread(image_path)

def calculate_image_hash(image_path):
    try:
        img = fix_image_orientation(image_path)
        if img is None:
            return None
        img_resized = cv2.resize(img, (256, 256))
        return hashlib.sha256(img_resized.tobytes()).hexdigest()
    except Exception as e:
        print(f"Error calculating hash for {image_path}: {e}")
        return None

def generate_thumbnail_path(content_hash):
    subdir = content_hash[:2]
    thumbnail_dir = os.path.join(app.config['THUMBNAILS_DIR'], subdir)
    os.makedirs(thumbnail_dir, exist_ok=True)
    return os.path.join(thumbnail_dir, f"{content_hash}.jpg")

def create_image_thumbnail(image_path, content_hash, max_width=THUMBNAIL_MAX_WIDTH, max_height=THUMBNAIL_MAX_HEIGHT):
    try:
        thumbnail_path = generate_thumbnail_path(content_hash)
        if os.path.exists(thumbnail_path):
            thumb_img = Image.open(thumbnail_path)
            thumb_width, thumb_height = thumb_img.size
            thumb_size = os.path.getsize(thumbnail_path)
            thumb_img.close()
            return thumbnail_path, thumb_width, thumb_height, thumb_size
        
        with Image.open(image_path) as img:
            img = ImageOps.exif_transpose(img)
            if img.mode != 'RGB':
                img = img.convert('RGB')
            
            original_width, original_height = img.size
            ratio = min(max_width / original_width, max_height / original_height)
            
            if ratio < 1.0:
                new_width = int(original_width * ratio)
                new_height = int(original_height * ratio)
                img = img.resize((new_width, new_height), Image.Resampling.LANCZOS)
            
            img.save(thumbnail_path, 'JPEG', quality=THUMBNAIL_QUALITY, optimize=True)
            thumb_width, thumb_height = img.size
            thumb_size = os.path.getsize(thumbnail_path)
            
            return thumbnail_path, thumb_width, thumb_height, thumb_size
    except Exception as e:
        print(f"Error creating thumbnail for {image_path}: {e}")
        return None, 0, 0, 0

def add_image_to_db(image_path):
    try:
        content_hash = calculate_image_hash(image_path)
        if not content_hash:
            return None
        
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT id FROM images WHERE content_hash = ?", (content_hash,))
        existing = cursor.fetchone()
        if existing:
            conn.close()
            return existing['id']
        
        corrected_img = fix_image_orientation(image_path)
        if corrected_img is None:
            conn.close()
            return None
        
        height, width = corrected_img.shape[:2]
        file_size = os.path.getsize(image_path)
        filename = os.path.basename(image_path)
        file_modified_time = os.path.getmtime(image_path)
        
        thumbnail_path, thumb_width, thumb_height, thumb_size = create_image_thumbnail(image_path, content_hash)
        has_thumbnail = thumbnail_path is not None
        
        cursor.execute('''INSERT INTO images 
            (path, filename, content_hash, width, height, file_size, file_modified_time, 
             has_thumbnail, thumbnail_path, thumbnail_width, thumbnail_height, thumbnail_size)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)''',
            (image_path, filename, content_hash, width, height, file_size,
             file_modified_time, has_thumbnail, thumbnail_path, thumb_width, thumb_height, thumb_size))
        
        image_id = cursor.lastrowid
        conn.commit()
        conn.close()
        return image_id
    except Exception as e:
        print(f"Error adding image to database: {e}")
        return None

# ==============================================================================
# 4. FACE THUMBNAIL GENERATION
# ==============================================================================
def generate_square_face_thumbnail_from_bbox(image_path, bbox, size=200, margin_factor=0.3):
    try:
        img = fix_image_orientation(image_path)
        if img is None:
            return None
        
        x1, y1, x2, y2 = bbox
        face_width = x2 - x1
        face_height = y2 - y1
        
        margin_w = int(face_width * margin_factor)
        margin_h = int(face_height * margin_factor)
        
        height, width = img.shape[:2]
        crop_x1 = max(0, x1 - margin_w)
        crop_y1 = max(0, y1 - margin_h)
        crop_x2 = min(width, x2 + margin_w)
        crop_y2 = min(height, y2 + margin_h)
        
        crop_width = crop_x2 - crop_x1
        crop_height = crop_y2 - crop_y1
        
        if crop_width > crop_height:
            diff = (crop_width - crop_height) // 2
            crop_y1 = max(0, crop_y1 - diff)
            crop_y2 = min(height, crop_y2 + diff)
        elif crop_height > crop_width:
            diff = (crop_height - crop_width) // 2
            crop_x1 = max(0, crop_x1 - diff)
            crop_x2 = min(width, crop_x2 + diff)
        
        face_crop = img[crop_y1:crop_y2, crop_x1:crop_x2]
        if face_crop.size == 0:
            return None
        
        face_pil = Image.fromarray(cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
        face_pil = face_pil.resize((size, size), Image.Resampling.LANCZOS)
        
        img_buffer = io.BytesIO()
        face_pil.save(img_buffer, format='JPEG', quality=90, optimize=True)
        return img_buffer.getvalue()
        
    except Exception as e:
        print(f"Error generating square face thumbnail: {e}")
        return None

def generate_face_thumbnail_from_bbox(image_path, bbox, max_size=200, margin=30):
    try:
        img = fix_image_orientation(image_path)
        if img is None:
            return None
        
        x1, y1, x2, y2 = bbox
        height, width = img.shape[:2]
        crop_x1 = max(0, x1 - margin)
        crop_y1 = max(0, y1 - margin)
        crop_x2 = min(width, x2 + margin)
        crop_y2 = min(height, y2 + margin)
        
        face_crop = img[crop_y1:crop_y2, crop_x1:crop_x2]
        if face_crop.size == 0:
            return None
        
        face_pil = Image.fromarray(cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB))
        face_pil.thumbnail((max_size, max_size), Image.Resampling.LANCZOS)
        
        img_buffer = io.BytesIO()
        face_pil.save(img_buffer, format='JPEG', quality=90, optimize=True)
        return img_buffer.getvalue()
        
    except Exception as e:
        print(f"Error generating face thumbnail: {e}")
        return None

# ==============================================================================
# 5. FACE DETECTION AND PROCESSING
# ==============================================================================
def assess_face_quality(face_data):
    quality_score = face_data.get('confidence', 0.5)
    bbox = face_data['bbox']
    
    face_width = bbox[2] - bbox[0]
    face_height = bbox[3] - bbox[1]
    face_area = face_width * face_height
    
    if face_area < 2500:
        quality_score *= 0.5
    
    aspect_ratio = face_width / face_height
    if aspect_ratio < 0.5 or aspect_ratio > 2.0:
        quality_score *= 0.7
    
    return quality_score

def process_single_image(args):
    image_path, image_directory = args
    
    try:
        full_path = os.path.join(image_directory, image_path)
        image_id = add_image_to_db(full_path)
        if not image_id:
            return []
        
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) as count FROM faces WHERE image_id = ?", (image_id,))
        existing_faces = cursor.fetchone()['count']
        conn.close()
        
        if existing_faces > 0:
            return []
        
        img = fix_image_orientation(full_path)
        if img is None:
            return []
        
        faces = face_app.get(img)
        faces_data = []
        
        for face_idx, face in enumerate(faces):
            bbox = face.bbox.astype(int)
            embedding = face.embedding / np.linalg.norm(face.embedding)
            confidence = face.det_score
            
            face_area = (bbox[2] - bbox[0]) * (bbox[3] - bbox[1])
            quality_score = assess_face_quality({'bbox': bbox, 'confidence': confidence})
            
            conn = get_db_connection()
            cursor = conn.cursor()
            cursor.execute('''INSERT INTO faces 
                (image_id, bbox_x1, bbox_y1, bbox_x2, bbox_y2, embedding, confidence, quality_score, face_area)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)''',
                (image_id, int(bbox[0]), int(bbox[1]), int(bbox[2]), int(bbox[3]),
                 embedding.tobytes(), confidence, quality_score, face_area))
            face_id = cursor.lastrowid
            conn.commit()
            conn.close()
            
            faces_data.append({
                'face_id': face_id, 'image_id': image_id, 'bbox': bbox, 'embedding': embedding,
                'confidence': confidence, 'quality_score': quality_score, 'path': full_path
            })
        
        with progress_lock:
            progress_data['thumbnails_created'] += 1
        
        return faces_data
        
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return []

def detect_faces_multithreaded(image_files, image_directory, num_threads=DEFAULT_THREAD_COUNT):
    update_progress(stage='face_detection', total=len(image_files), 
                   text=f'Detecting faces in {len(image_files)} images with orientation correction...')
    
    all_faces_data = []
    processed_count = 0
    args_list = [(img_file, image_directory) for img_file in image_files]
    
    with ThreadPoolExecutor(max_workers=num_threads) as executor:
        future_to_image = {executor.submit(process_single_image, args): args[0] for args in args_list}
        
        for future in as_completed(future_to_image):
            image_file = future_to_image[future]
            try:
                faces_data = future.result()
                all_faces_data.extend(faces_data)
                processed_count += 1
                
                update_progress(current=processed_count, images_processed=processed_count,
                               faces_detected=len(all_faces_data), current_image=image_file,
                               text=f'Processed {processed_count} / {len(image_files)} images | '
                                   f'Faces detected: {len(all_faces_data)}')
                
            except Exception as e:
                print(f"Error processing {image_file}: {e}")
                processed_count += 1
    
    return all_faces_data

# ==============================================================================
# 6. CLUSTERING FUNCTIONS
# ==============================================================================
def get_all_unassigned_faces():
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute('''SELECT f.id, f.embedding, f.quality_score, f.confidence, f.image_id
        FROM faces f LEFT JOIN person_faces pf ON f.id = pf.face_id
        WHERE pf.face_id IS NULL AND f.quality_score >= 0.3 ORDER BY f.quality_score DESC''')
    results = cursor.fetchall()
    conn.close()
    
    faces_data = []
    for row in results:
        embedding = np.frombuffer(row['embedding'], dtype=np.float32)
        faces_data.append({
            'face_id': row['id'], 'image_id': row['image_id'], 'embedding': embedding,
            'quality_score': row['quality_score'], 'confidence': row['confidence']
        })
    return faces_data

def cluster_faces_dbscan(faces_data, eps, min_samples):
    if not faces_data:
        return
    
    update_progress(stage='clustering', text='Performing DBSCAN clustering...')
    
    embeddings = np.array([face['embedding'] for face in faces_data])
    face_ids = [face['face_id'] for face in faces_data]
    image_ids = [face['image_id'] for face in faces_data]
    
    clustering = DBSCAN(metric="cosine", eps=eps, min_samples=min_samples, n_jobs=-1)
    labels = clustering.fit_predict(embeddings)
    
    unique_labels = set(labels)
    num_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    
    conn = get_db_connection()
    cursor = conn.cursor()
    
    for label in unique_labels:
        if label == -1:
            continue
        
        cluster_face_indices = np.where(labels == label)[0]
        cluster_face_ids = [face_ids[i] for i in cluster_face_indices]
        cluster_image_ids = [image_ids[i] for i in cluster_face_indices]
        cluster_embeddings = [embeddings[i] for i in cluster_face_indices]
        
        centroid = np.mean(cluster_embeddings, axis=0)
        centroid = centroid / np.linalg.norm(centroid)
        
        distances = [cosine(emb, centroid) for emb in cluster_embeddings]
        rep_face_idx = cluster_face_indices[np.argmin(distances)]
        rep_face_id = face_ids[rep_face_idx]
        rep_image_id = image_ids[rep_face_idx]
        
        person_name = f"Person {label + 1:03d}"
        cursor.execute('''INSERT INTO persons 
            (name, representative_face_id, representative_image_id, face_count, centroid_embedding)
            VALUES (?, ?, ?, ?, ?)''',
            (person_name, rep_face_id, rep_image_id, len(cluster_face_ids), centroid.tobytes()))
        person_id = cursor.lastrowid
        
        for i, face_id in enumerate(cluster_face_ids):
            distance_to_centroid = cosine(cluster_embeddings[i], centroid)
            cursor.execute('''INSERT INTO person_faces (person_id, face_id, distance_to_centroid)
                VALUES (?, ?, ?)''', (person_id, face_id, distance_to_centroid))
    
    conn.commit()
    conn.close()
    update_progress(clusters_formed=num_clusters, text=f'Clustering complete: {num_clusters} persons identified')

# ==============================================================================
# 7. PROGRESS TRACKING
# ==============================================================================
def update_progress(stage=None, current=None, total=None, text=None, **kwargs):
    with progress_lock:
        if stage is not None:
            progress_data['current_stage'] = stage
        if current is not None:
            progress_data['current'] = current
        if total is not None:
            progress_data['total'] = total
        if text is not None:
            progress_data['text'] = text
        
        for key, value in kwargs.items():
            if key in progress_data:
                progress_data[key] = value
        
        if progress_data['start_time']:
            progress_data['elapsed_time'] = time.time() - progress_data['start_time']
            if progress_data['total'] > 0 and progress_data['current'] > 0:
                progress_ratio = progress_data['current'] / progress_data['total']
                if progress_ratio > 0:
                    total_estimated_time = progress_data['elapsed_time'] / progress_ratio
                    progress_data['eta_seconds'] = max(0, total_estimated_time - progress_data['elapsed_time'])
                    progress_data['processing_speed'] = progress_data['current'] / progress_data['elapsed_time']

def reset_progress():
    with progress_lock:
        progress_data.update({
            'current': 0, 'total': 0, 'faces_detected': 0, 'images_processed': 0,
            'faces_processed': 0, 'clusters_formed': 0, 'suggestions_generated': 0,
            'thumbnails_created': 0, 'start_time': time.time(), 'elapsed_time': 0,
            'current_stage': 'starting', 'processing_speed': 0, 'eta_seconds': 0,
            'is_processing': True, 'current_image': ''
        })

# ==============================================================================
# 8. SUGGESTION GENERATION
# ==============================================================================
def generate_suggestions():
    global suggestions_cache
    suggestions_cache = []
    
    conn = get_db_connection()
    cursor = conn.cursor()
    
    cursor.execute('''SELECT f.id, f.embedding, f.image_id, f.bbox_x1, f.bbox_y1, f.bbox_x2, f.bbox_y2, 
        i.path, i.content_hash FROM faces f JOIN images i ON f.image_id = i.id
        LEFT JOIN person_faces pf ON f.id = pf.face_id
        WHERE pf.face_id IS NULL AND f.quality_score >= 0.3 ORDER BY f.quality_score DESC LIMIT 50''')
    unassigned_faces = cursor.fetchall()
    
    cursor.execute('''SELECT p.id, p.name, p.centroid_embedding, p.representative_face_id
        FROM persons p WHERE p.face_count >= 2''')
    persons = cursor.fetchall()
    
    conn.close()
    
    if not unassigned_faces or not persons:
        return
    
    eps = model_state['eps']
    suggestion_eps = model_state['suggestion_eps']
    suggestions_generated = 0
    
    for face_row in unassigned_faces:
        face_embedding = np.frombuffer(face_row['embedding'], dtype=np.float32)
        
        best_person = None
        min_distance = float('inf')
        
        for person_row in persons:
            person_centroid = np.frombuffer(person_row['centroid_embedding'], dtype=np.float32)
            distance = cosine(face_embedding, person_centroid)
            
            if distance < min_distance:
                min_distance = distance
                best_person = person_row
        
        if best_person and eps < min_distance < suggestion_eps:
            try:
                suggestions_cache.append({
                    'face_id': face_row['id'], 'person_id': best_person['id'],
                    'person_name': best_person['name'], 'distance': min_distance,
                    'unassigned_face_url': f"/face_thumbnail/{face_row['id']}",
                    'representative_face_url': f"/face_thumbnail/{best_person['representative_face_id']}"
                })
                suggestions_generated += 1
                
                if suggestions_generated >= 20:
                    break
            except Exception as e:
                print(f"Error creating suggestion: {e}")
                continue
    
    update_progress(suggestions_generated=suggestions_generated)

def calculate_model_accuracy():
    if len(feedback_history) < 10:
        return 0.5
    
    correct_predictions = 0
    total_predictions = 0
    current_eps = model_state['eps']
    
    for feedback in feedback_history[-50:]:
        predicted_decision = 'yes' if feedback['distance'] <= current_eps else 'no'
        if predicted_decision == feedback['decision']:
            correct_predictions += 1
        total_predictions += 1
    
    return correct_predictions / total_predictions if total_predictions > 0 else 0.5

# ==============================================================================
# 9. MAIN PROCESSING FUNCTION
# ==============================================================================
def process_classification_job(image_directory, suggestion_eps, min_samples, num_threads):
    global progress_data, model_state, processing_thread
    
    try:
        reset_progress()
        eps = model_state['eps']
        
        update_progress(stage='scanning', text='Scanning directory for images...')
        image_files = [f for f in os.listdir(image_directory) 
                      if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tiff'))]
        
        if not image_files:
            update_progress(text='No supported image files found.', is_processing=False)
            return
        
        all_faces_data = detect_faces_multithreaded(image_files, image_directory, num_threads)
        
        if not all_faces_data:
            conn = get_db_connection()
            cursor = conn.cursor()
            cursor.execute("SELECT COUNT(*) as count FROM faces WHERE quality_score >= 0.3")
            existing_faces = cursor.fetchone()['count']
            conn.close()
            
            if existing_faces == 0:
                update_progress(text='No faces detected in any images.', is_processing=False)
                return
        
        unassigned_faces = get_all_unassigned_faces()
        
        if unassigned_faces:
            cluster_faces_dbscan(unassigned_faces, eps, min_samples)
        
        update_progress(stage='suggestions', text='Generating intelligent suggestions...')
        generate_suggestions()
        
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT COUNT(*) as count FROM persons")
        num_persons = cursor.fetchone()['count']
        cursor.execute("SELECT COUNT(*) as count FROM faces f LEFT JOIN person_faces pf ON f.id = pf.face_id WHERE pf.face_id IS NULL")
        num_unassigned = cursor.fetchone()['count']
        cursor.execute("SELECT COUNT(*) as count FROM images WHERE has_thumbnail = TRUE")
        num_thumbnails = cursor.fetchone()['count']
        conn.close()
        
        update_progress(stage='complete',
            text=f'Complete! {num_persons} persons identified, {len(suggestions_cache)} suggestions generated, {num_thumbnails} thumbnails created, {num_unassigned} unassigned faces.',
            is_processing=False)
        
    except Exception as e:
        print(f"Error in classification job: {e}")
        update_progress(text=f'Error: {str(e)}', is_processing=False)
    finally:
        with progress_lock:
            progress_data['is_processing'] = False
        processing_thread = None

# ==============================================================================
# 10. DATABASE QUERY FUNCTIONS
# ==============================================================================
def get_database_stats():
    conn = get_db_connection()
    cursor = conn.cursor()
    
    cursor.execute("SELECT COUNT(*) as count FROM images")
    total_images = cursor.fetchone()['count']
    cursor.execute("SELECT COUNT(*) as count FROM faces")
    total_faces = cursor.fetchone()['count']
    cursor.execute("SELECT COUNT(*) as count FROM persons")
    total_persons = cursor.fetchone()['count']
    cursor.execute("SELECT COUNT(*) as count FROM images WHERE has_thumbnail = TRUE")
    total_thumbnails = cursor.fetchone()['count']
    
    conn.close()
    return {'total_images': total_images, 'total_faces': total_faces, 
            'total_persons': total_persons, 'total_thumbnails': total_thumbnails}

def get_persons_summary():
    conn = get_db_connection()
    cursor = conn.cursor()
    
    cursor.execute('''SELECT p.id, p.name, p.face_count, p.representative_face_id, p.representative_image_id,
        AVG(f.quality_score) as avg_quality FROM persons p
        LEFT JOIN person_faces pf ON p.id = pf.person_id
        LEFT JOIN faces f ON pf.face_id = f.id
        GROUP BY p.id, p.name, p.face_count, p.representative_face_id, p.representative_image_id
        ORDER BY p.face_count DESC, p.name''')
    persons = cursor.fetchall()
    
    result = []
    for person in persons:
        result.append({
            'id': person['id'], 'name': person['name'], 'face_count': person['face_count'],
            'representative_face_cropped_url': f"/person_face_thumbnail/{person['id']}",
            'avg_quality': person['avg_quality'] or 0
        })
    
    conn.close()
    return result

def get_person_with_gallery(person_id):
    conn = get_db_connection()
    cursor = conn.cursor()
    
    cursor.execute('SELECT * FROM persons WHERE id = ?', (person_id,))
    person = cursor.fetchone()
    if not person:
        conn.close()
        return None
    
    cursor.execute('''SELECT DISTINCT i.id, i.filename, i.content_hash, i.has_thumbnail, 
        i.thumbnail_width, i.thumbnail_height, i.file_modified_time, COUNT(pf.face_id) as face_count,
        AVG(f.quality_score) as avg_quality FROM images i
        JOIN faces f ON i.id = f.image_id JOIN person_faces pf ON f.id = pf.face_id
        WHERE pf.person_id = ? GROUP BY i.id, i.filename, i.content_hash, i.has_thumbnail, 
        i.thumbnail_width, i.thumbnail_height, i.file_modified_time
        ORDER BY i.file_modified_time DESC, avg_quality DESC''', (person_id,))
    images_data = cursor.fetchall()
    
    cursor.execute('''SELECT i.content_hash, i.has_thumbnail FROM images i WHERE i.id = ?''', 
                  (person['representative_image_id'],))
    rep_result = cursor.fetchone()
    
    representative_image = None
    if rep_result and rep_result['has_thumbnail']:
        representative_image = {'thumbnail_url': f"/image_thumbnail/{rep_result['content_hash']}"}
    
    images_gallery = []
    total_quality = 0
    for img in images_data:
        if img['has_thumbnail']:
            images_gallery.append({
                'thumbnail_url': f"/image_thumbnail/{img['content_hash']}",
                'original_url': f"/original_image/{img['content_hash']}",
                'filename': img['filename'], 'face_count': img['face_count'],
                'quality': img['avg_quality'],
                'thumbnail_width': img['thumbnail_width'] or 300,
                'thumbnail_height': img['thumbnail_height'] or 300,
                'file_modified_time': img['file_modified_time']
            })
            total_quality += img['avg_quality']
    
    avg_quality = total_quality / len(images_gallery) if images_gallery else 0
    
    result = {
        'id': person['id'], 'name': person['name'], 'face_count': person['face_count'],
        'representative_image': representative_image, 'images': images_gallery, 'avg_quality': avg_quality
    }
    
    conn.close()
    return result

# ==============================================================================
# 11. HTML TEMPLATES
# ==============================================================================

HOME_PAGE_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>📼️ Face Clustering - Person Gallery</title>
    <style>
        * { 
            box-sizing: border-box; 
        }
        
        body { 
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; 
            margin: 0; 
            background-color: #f4f7f6; 
            color: #333; 
        }
        
        .container { 
            width: 95%; 
            max-width: 1400px;
            margin: 20px auto; 
            padding: 0; 
        }
        
        .header { 
            text-align: center; 
            border-bottom: 1px solid #ddd; 
            padding-bottom: 20px; 
            margin-bottom: 20px; 
        }
        
        h1, h2, h3 { 
            color: #333; 
        }
        
        .panel { 
            background-color: #fff; 
            padding: 20px; 
            border-radius: 8px; 
            box-shadow: 0 2px 4px rgba(0,0,0,0.1); 
            margin-bottom: 20px; 
        }
        
        .form-group { 
            display: flex; 
            flex-direction: column; 
            margin-bottom: 15px; 
        }
        
        .form-group label { 
            font-weight: bold; 
            margin-bottom: 5px; 
        }
        
        .form-group input { 
            padding: 10px; 
            border: 1px solid #ccc; 
            border-radius: 4px; 
            font-size: 1em; 
        }
        
        .btn { 
            padding: 10px 15px; 
            border: none; 
            border-radius: 4px; 
            cursor: pointer; 
            font-weight: bold; 
            text-align: center; 
            text-decoration: none; 
            display: inline-block; 
            transition: background-color 0.2s; 
        }
        
        .btn-primary { 
            background-color: #007bff; 
            color: white; 
            width: 100%; 
        }
        
        .btn-danger { 
            background-color: #dc3545; 
            color: white; 
        }
        
        /* Model stats grid */
        .model-stats { 
            display: grid; 
            grid-template-columns: repeat(auto-fit, minmax(180px, 1fr)); 
            gap: 15px; 
            margin-bottom: 20px;
        }
        
        .stat-card { 
            background-color: #f8f9fa;
            padding: 15px; 
            border-radius: 8px; 
            text-align: center; 
            border-left: 4px solid #007bff; 
        }
        
        .stat-value { 
            font-size: 1.5em; 
            font-weight: bold; 
            color: #007bff; 
        }
        
        .stat-label { 
            font-size: 0.9em;
            color: #666; 
            margin-top: 5px; 
        }
        
        /* Flash messages */
        .flash-message { 
            padding: 15px; 
            margin-bottom: 20px; 
            border-radius: 4px; 
            text-align: center; 
            font-weight: bold; 
        }
        
        .flash-info { 
            color: #0c5460; 
            background-color: #d1ecf1; 
            border-color: #bee5eb; 
        }
        
        .flash-error { 
            color: #721c24; 
            background-color: #f8d7da; 
            border-color: #f5c6cb; 
        }
        
        .flash-success { 
            color: #155724; 
            background-color: #d4edda; 
            border-color: #c3e6cb; 
        }
        
        /* UPDATED :: Dynamic, responsive square thumbnail grid */
        .persons-grid {
            display: grid;
            /* This single line handles responsiveness, min/max size, and fitting */
            grid-template-columns: repeat(auto-fit, minmax(160px, 1fr));
            gap: 2px; /* Maintains the 2px margin horizontally and vertically */
            padding: 20px;
            background: transparent;
            width: 100vw;
            margin-left: calc(-50vw + 50%);
            padding-left: 50px;
            padding-right: 50px;
        }

        .person-thumbnail-container {
            position: relative;
            width: 100%; /* Occupy the full grid cell width */
            aspect-ratio: 1 / 1; /* Enforce a square shape */
            border-radius: 0;
            overflow: hidden;
            cursor: pointer;
            transition: transform 0.2s, box-shadow 0.2s;
            box-shadow: 0 4px 12px rgba(0,0,0,0.15);
        }

        .person-thumbnail-container:hover {
            transform: translateY(-4px);
            box-shadow: 0 8px 25px rgba(0,0,0,0.2);
        }

        .person-thumbnail {
            width: 100%;
            height: 100%;
            object-fit: cover;
            object-position: center center;
            display: block;
            border-radius: 0;
        }

        .person-label-overlay {
            position: absolute;
            bottom: 0;
            left: 0;
            width: 100%;
            padding: 12px 8px;
            text-align: center;
            color: white;
            font-weight: bold;
            font-size: 1.1em;
            background: linear-gradient(transparent, rgba(0, 0, 0, 0.7));
            pointer-events: none;
            text-shadow: 0 1px 3px rgba(0,0,0,0.8);
        }

        .person-thumbnail-container:hover .person-label-overlay {
            background: linear-gradient(transparent, rgba(0, 0, 0, 0.9));
        }
        
        /* Suggestion cards */
        .suggestion-stack-container { 
            position: relative; 
            height: 400px; 
            max-width: 600px; 
            margin: 20px auto; 
        }
        
        .suggestion-card { 
            display: flex; 
            flex-direction: column; 
            justify-content: center; 
            align-items: center; 
            background-color: white; 
            border: 1px solid #ccc; 
            border-radius: 15px; 
            box-shadow: 0 4px 8px rgba(0,0,0,0.1); 
            position: absolute; 
            width: 100%; 
            height: 100%; 
            transition: transform 0.4s ease, opacity 0.4s ease; 
            user-select: none; 
        }
        
        .card-content { 
            display: flex; 
            justify-content: space-around; 
            align-items: center; 
            width: 100%; 
            padding: 10px; 
            gap: 2px; 
        }
        
        .card-content img { 
            width: calc(45% - 1px); 
            max-height: 250px; 
            border-radius: 8px; 
            object-fit: cover;
            margin: 2px; 
        }
        
        .card-question { 
            font-weight: bold; 
            font-size: 1.2em; 
            margin-bottom: 10px; 
        }
        
        .card-distance { 
            font-size: 0.9em; 
            color: #666; 
            margin-bottom: 15px; 
        }
        
        .card-controls { 
            display: flex; 
            justify-content: space-around; 
            width: 100%; 
            padding: 20px; 
        }
        
        .card-controls .btn { 
            font-size: 1.5em; 
            border-radius: 50%; 
            width: 70px; 
            height: 70px; 
            display: flex; 
            justify-content: center; 
            align-items: center; 
        }
        
        .btn-no { 
            background-color: #dc3545; 
            color: white; 
        }
        
        .btn-yes { 
            background-color: #28a745; 
            color: white; 
        }
        
        .btn-neutral { 
            background-color: #6c757d; 
            color: white; 
        }
        
        .card-is-active { 
            transform: translate(0, 0) scale(1); 
            opacity: 1; 
            z-index: 10; 
        }
        
        .card-is-next { 
            transform: translate(0, 20px) scale(0.95); 
            opacity: 1; 
            z-index: 9; 
        }
        
        .card-is-gone-left { 
            transform: translateX(-150%) rotate(-30deg); 
            opacity: 0; 
        }
        
        .card-is-gone-right { 
            transform: translateX(150%) rotate(30deg); 
            opacity: 0; 
        }
        
        .card-is-gone-down { 
            transform: translateY(150%); 
            opacity: 0; 
        }
        
        /* Loading overlay */
        .loading-overlay {
            position: fixed;
            top: 0;
            left: 0;
            width: 100%;
            height: 100%;
            background: rgba(0, 0, 0, 0.8);
            color: white;
            display: none;
            align-items: center;
            justify-content: center;
            z-index: 1000;
            text-align: center;
        }
        
        .loading-content {
            background: rgba(20, 20, 20, 0.9);
            padding: 40px;
            border-radius: 10px;
            max-width: 500px;
        }
        
        .progress-stats {
            display: grid;
            grid-template-columns: repeat(2, 1fr);
            gap: 20px;
            margin: 20px 0;
        }
        
        .progress-stat {
            text-align: center;
        }
        
        .progress-stat .value {
            font-size: 2em;
            font-weight: bold;
            color: #10b981;
        }
        
        .progress-stat .label {
            font-size: 0.9em;
            color: #888;
        }
        
        /* Responsive design adjustments */
        @media (max-width: 900px) {
            .container { 
                width: 98%; 
                margin: 10px auto; 
            }
            
            .model-stats { 
                grid-template-columns: repeat(auto-fit, minmax(150px, 1fr)); 
                gap: 10px; 
            }
            
            .persons-grid { 
                /* The grid is already responsive, just adjust padding for smaller screens */
                padding-left: 20px;
                padding-right: 20px;
            }
        }
        
        @media (max-width: 600px) {
            .model-stats { 
                grid-template-columns: repeat(2, 1fr); 
                gap: 8px; 
            }
            
            .suggestion-stack-container { 
                height: 350px; 
                max-width: 95%; 
            }
            
            .card-content { 
                flex-direction: column; 
                gap: 2px; 
            }
            
            .card-content img { 
                width: calc(80% - 4px); 
                max-height: 120px;
                margin: 2px;
            }
            
            .progress-stats { 
                grid-template-columns: 1fr; 
                gap: 15px; 
            }
            
            .persons-grid { 
                /* The grid's minmax() function handles smaller sizes automatically.
                   Just reduce the horizontal padding. */
                padding-left: 10px;
                padding-right: 10px;
            }
            
            .person-label-overlay {
                font-size: 0.9em;
                padding: 8px 4px;
            }
        }
    </style>
</head>
<body>
    <div id="loadingOverlay" class="loading-overlay">
        <div class="loading-content">
            <h2 id="progressTitle">📼️ Orientation-Corrected Face Clustering</h2>
            <p id="progressSubtitle">Processing images with EXIF orientation correction...</p>
            
            <div class="progress-stats">
                <div class="progress-stat">
                    <div class="value" id="facesDetected">0</div>
                    <div class="label">Faces Detected</div>
                </div>
                <div class="progress-stat">
                    <div class="value" id="imagesProcessed">0</div>
                    <div class="label">Images Processed</div>
                </div>
                <div class="progress-stat">
                    <div class="value" id="thumbnailsCreated">0</div>
                    <div class="label">Thumbnails Created</div>
                </div>
                <div class="progress-stat">
                    <div class="value" id="etaTime">--</div>
                    <div class="label">ETA</div>
                </div>
            </div>
            
            <div id="currentStatus">
                <p><strong>Current:</strong> <span id="currentImageName">Initializing...</span></p>
                <p><strong>Stage:</strong> <span id="currentImageStatus">Starting...</span></p>
            </div>
        </div>
    </div>
    
    <div class="container">
        <div class="header">
            <h1>📼️ Face Clustering - Person Gallery</h1>
            <p style="opacity: 0.7;">Smart face recognition with orientation correction and square thumbnails</p>
        </div>
        
        {% with messages = get_flashed_messages(with_categories=true) %}
        {% if messages %}
            {% for category, message in messages %}
            <div class="flash-message flash-{{ category }}">{{ message }}</div>
            {% endfor %}
        {% endif %}
        {% endwith %}
        
        <div class="panel">
            <div class="model-stats">
                <div class="stat-card">
                    <div class="stat-value">{{ "%.4f"|format(model_state.eps) }}</div>
                    <div class="stat-label">Current EPS</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">{{ model_state.total_decisions }}</div>
                    <div class="stat-label">Total Decisions</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">{{ "%.1f"|format(model_accuracy * 100) }}%</div>
                    <div class="stat-label">Model Accuracy</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">{{ database_stats.total_images }}</div>
                    <div class="stat-label">Total Images</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">{{ database_stats.total_faces }}</div>
                    <div class="stat-label">Total Faces</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">{{ database_stats.total_persons }}</div>
                    <div class="stat-label">Persons Identified</div>
                </div>
                <div class="stat-card">
                    <div class="stat-value">{{ database_stats.total_thumbnails }}</div>
                    <div class="stat-label">Corrected Thumbnails</div>
                </div>
            </div>
            
            <form action="/classify" method="post" id="classifyForm">
                <p style="text-align:center;">Configure parameters for face clustering with square thumbnail layout.</p>
                <div class="form-group">
                    <label for="image_directory">Full Path to Photo Directory</label>
                    <input type="text" id="image_directory" name="image_directory" value="{{ image_directory }}" required>
                </div>
                <div style="display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 20px;">
                    <div class="form-group">
                        <label for="suggestion_eps">Suggestion Threshold</label>
                        <input type="number" id="suggestion_eps" name="suggestion_eps" value="{{ model_state.suggestion_eps }}" step="0.01" required>
                    </div>
                    <div class="form-group">
                        <label for="min_samples">Min Photos per Group</label>
                        <input type="number" id="min_samples" name="min_samples" value="{{ min_samples }}" step="1" required>
                    </div>
                    <div class="form-group">
                        <label for="num_threads">Processing Threads</label>
                        <input type="number" id="num_threads" name="num_threads" value="{{ num_threads }}" min="1" max="16" step="1" required>
                    </div>
                </div>
                <div class="form-group">
                    <button type="submit" class="btn btn-primary">📼️ Run Face Classification</button>
                </div>
            </form>
        </div>
        
        {% if progress_data.is_processing %}
        <div class="panel">
            <h3 style="text-align: center; color: #007bff;">⚡ Processing in Progress...</h3>
            <p style="text-align: center;">Classification with orientation correction is running in the background. You can refresh this page to see updated results.</p>
        </div>
        {% endif %}
        
        {% if suggestions %}
        <div class="panel suggestions-area">
            <h2 id="suggestionCounter"></h2>
            <div id="suggestionStack" class="suggestion-stack-container">
                {% for suggestion in suggestions %}
                <div class="suggestion-card" data-suggestion-id="{{ loop.index0 }}">
                    <p class="card-question">Are these the same person?</p>
                    <p class="card-distance">Distance: {{ "%.4f"|format(suggestion.distance) }}</p>
                    <div class="card-content">
                        <img src="{{ suggestion.unassigned_face_url }}" alt="Unassigned Face">
                        <img src="{{ suggestion.representative_face_url }}" alt="Known Person">
                    </div>
                    <div class="card-controls">
                        <button type="button" class="btn btn-no" onclick="makeDecision({{ loop.index0 }}, 'no')">&times;</button>
                        <button type="button" class="btn btn-neutral" onclick="makeDecision({{ loop.index0 }}, 'not_sure')">?</button>
                        <button type="button" class="btn btn-yes" onclick="makeDecision({{ loop.index0 }}, 'yes')">✓</button>
                    </div>
                </div>
                {% endfor %}
            </div>
            <p id="suggestionsComplete" style="text-align:center; display:none; font-weight:bold;">
                All suggestions reviewed! The model has learned from your decisions. Refreshing...
            </p>
        </div>
        {% endif %}
        
        <div class="panel" style="text-align: center;">
            <form action="/reset_model" method="post" style="display: inline-block; margin-right: 10px;">
                <button type="submit" class="btn btn-danger" onclick="return confirm('Reset model to default settings?');">Reset Model Only</button>
            </form>
            <form action="/clear" method="post" style="display: inline-block;">
                <button type="submit" class="btn btn-danger" onclick="return confirm('Delete all database data and thumbnails?');">Clear All Data</button>
            </form>
        </div>
    </div>
        
    <div class="results-area">
        <h2 style="text-align: center; margin-bottom: 30px;">📼️ Identified Persons</h2>
        {% if persons_summary %}
            <div class="persons-grid">
                {% for person in persons_summary %}
                <div class="person-thumbnail-container" onclick="location.href='/person/{{ person.id }}'">
                    <img src="{{ person.representative_face_cropped_url }}" alt="{{ person.name }}" class="person-thumbnail">
                    <div class="person-label-overlay">{{ person.name }}</div>
                </div>
                {% endfor %}
            </div>
        {% else %}
            <p style="text-align:center; padding: 40px; color: #666;">No persons identified yet. Run a classification to see results.</p>
        {% endif %}
    </div>
    
    <script>
        let progressInterval;
        
        // Form submission with loading overlay
        const classifyForm = document.getElementById('classifyForm');
        if (classifyForm) {
            classifyForm.addEventListener('submit', (event) => {
                event.preventDefault();
                document.getElementById('loadingOverlay').style.display = 'flex';
                progressInterval = setInterval(updateProgress, 500);
                
                const formData = new FormData(classifyForm);
                fetch('/classify', { method: 'POST', body: formData })
                .then(response => {
                    if (!response.ok) return response.json().then(err => { throw new Error(err.message) });
                    return response.json();
                })
                .then(data => {
                    if (data.status === 'success') console.log('Classification started successfully.');
                    else throw new Error(data.message || 'Failed to start classification.');
                })
                .catch(error => {
                    console.error('Error starting classification:', error);
                    alert('Error: ' + error.message);
                    clearInterval(progressInterval);
                    document.getElementById('loadingOverlay').style.display = 'none';
                });
            });
        }
        
        // Progress update function
        function updateProgress() {
            fetch('/progress')
            .then(r => r.json())
            .then(data => {
                document.getElementById('facesDetected').textContent = data.faces_detected || 0;
                document.getElementById('imagesProcessed').textContent = data.images_processed || 0;
                document.getElementById('thumbnailsCreated').textContent = data.thumbnails_created || 0;
                
                if (data.eta_seconds > 0) {
                    const minutes = Math.floor(data.eta_seconds / 60);
                    const seconds = Math.round(data.eta_seconds % 60);
                    document.getElementById('etaTime').textContent = minutes > 0 ? `${minutes}m ${seconds}s` : `${seconds}s`;
                } else {
                    document.getElementById('etaTime').textContent = '--';
                }

                const imgName = data.current_image || 'Waiting for task...';
                document.getElementById('currentImageName').textContent = imgName;
                document.getElementById('currentImageStatus').textContent = data.current_stage.replace(/_/g, ' ');
                
                if (!data.is_processing && data.start_time) {
                    clearInterval(progressInterval);
                    
                    document.getElementById('progressTitle').textContent = 'Processing Complete!';
                    document.getElementById('progressSubtitle').textContent = data.text;
                    
                    setTimeout(() => {
                        document.getElementById('loadingOverlay').style.display = 'none';
                        window.location.reload();
                    }, 2000);
                }
            })
            .catch(err => {
                console.error('Progress polling failed:', err);
                clearInterval(progressInterval);
            });
        }
        
        // Suggestion card functionality
        const suggestionCards = document.querySelectorAll('.suggestion-card');
        let currentCardIndex = 0;
        
        function updateCardStack() {
            const counter = document.getElementById('suggestionCounter');
            if (counter) {
                if (currentCardIndex >= suggestionCards.length) {
                    counter.textContent = 'All suggestions reviewed!';
                    document.getElementById('suggestionsComplete').style.display = 'block';
                    setTimeout(() => window.location.reload(), 2000);
                } else {
                    counter.textContent = `Suggestion ${currentCardIndex + 1} of ${suggestionCards.length}`;
                }
            }
            
            suggestionCards.forEach((card, index) => {
                card.classList.remove('card-is-active', 'card-is-next', 'card-is-gone-left', 'card-is-gone-right', 'card-is-gone-down');
                if (index === currentCardIndex) {
                    card.classList.add('card-is-active');
                } else if (index === currentCardIndex + 1) {
                    card.classList.add('card-is-next');
                }
            });
        }
        
        function makeDecision(cardIndex, decision) {
            const card = document.querySelector(`.suggestion-card[data-suggestion-id='${cardIndex}']`);
            if (!card) return;
            
            if (decision === 'yes') card.classList.add('card-is-gone-right');
            else if (decision === 'no') card.classList.add('card-is-gone-left');
            else card.classList.add('card-is-gone-down');
            
            fetch('/decide', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({
                    suggestion_id: cardIndex,
                    decision: decision,
                    confidence: 1.0,
                    timestamp: Date.now()
                })
            });
            
            currentCardIndex++;
            setTimeout(updateCardStack, 100);
        }
        
        // Keyboard shortcuts for suggestions
        document.addEventListener('keydown', e => {
            if (suggestionCards.length > 0 && currentCardIndex < suggestionCards.length) {
                if (e.key === 'ArrowRight' || e.key.toLowerCase() === 'y') makeDecision(currentCardIndex, 'yes');
                else if (e.key === 'ArrowLeft' || e.key.toLowerCase() === 'n') makeDecision(currentCardIndex, 'no');
                else if (e.key === 'ArrowDown' || e.key.toLowerCase() === 'u') makeDecision(currentCardIndex, 'not_sure');
            }
        });
        
        // Initialize suggestion cards
        if (suggestionCards.length > 0) {
            updateCardStack();
        }
    </script>
</body>
</html>"""



PERSON_PAGE_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>📼️ {{ person.name }} - Person Gallery</title>
    <style>
        * { 
            box-sizing: border-box; 
        }
        
        body { 
            font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif; 
            margin: 0; 
            background-color: #f4f7f6; 
            color: #333; 
        }
        
        .container { 
            width: 95%; 
            max-width: 1400px;
            margin: 20px auto; 
            padding: 0; 
        }
        
        .header { 
            text-align: center; 
            border-bottom: 1px solid #ddd; 
            padding-bottom: 20px; 
            margin-bottom: 20px; 
        }
        
        h1, h2, h3 { 
            color: #333; 
        }
        
        .back-button {
            display: inline-block;
            padding: 10px 20px;
            background-color: #007bff;
            color: white;
            text-decoration: none;
            border-radius: 4px;
            margin-bottom: 20px;
            font-weight: bold;
            transition: background-color 0.2s;
        }
        
        .back-button:hover {
            background-color: #0056b3;
        }
        
        .person-info {
            background-color: #fff;
            padding: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
            margin-bottom: 20px;
            display: flex;
            align-items: center;
            gap: 20px;
        }
        
        .person-representative {
            width: 100px;
            height: 100px;
            border-radius: 50%;
            object-fit: cover;
            border: 4px solid #007bff;
            flex-shrink: 0;
        }
        
        .person-details h2 {
            margin: 0 0 15px 0;
            font-size: 2em;
        }
        
        .person-stats {
            display: grid;
            grid-template-columns: repeat(3, 1fr);
            gap: 20px;
            margin: 10px 0;
        }
        
        .stat-item {
            text-align: center;
            padding: 10px;
            background-color: #f8f9fa;
            border-radius: 8px;
        }
        
        .stat-value {
            font-size: 1.5em;
            font-weight: bold;
            color: #007bff;
        }
        
        .stat-label {
            font-size: 0.9em;
            color: #666;
            margin-top: 5px;
        }
        
        /* Gallery area with full-width layout and 2px margins */
        .gallery-area {
            background: transparent !important;
            box-shadow: none !important;
            border: none !important;
            padding: 0 50px !important;
            width: 100vw !important;
            margin-left: calc(-50vw + 50%) !important;
            margin-bottom: 20px;
        }
        
        .gallery-area h2 {
            text-align: center;
            margin-bottom: 30px;
        }
        
        /* Date group styling with shrink-wrap capability and 2px margins */
        .date-group {
            margin-bottom: 25px;
            margin-right: 50px;
            display: block;
            width: 100%;
        }
        
        .date-group.single-row {
            display: inline-block;
            width: auto;
            max-width: 100%;
            margin-right: 50px;
        }
        
        .date-group.multi-row {
            display: block;
            width: 100%;
            margin-right: 0;
        }
        
        .date-group:last-child {
            margin-right: 0;
        }
        
        .date-header {
            font-weight: bold;
            font-size: 1.2em;
            color: #333;
            margin: 5px 0 10px 0;
            background: none !important;
            box-shadow: none !important;
            padding: 0 !important;
            border: none !important;
            border-radius: 0 !important;
        }
        
        /* Google Photos style perfectly justified gallery with 2px gaps */
        .justified-gallery {
            display: flex;
            flex-direction: column;
            gap: 2px; /* 2px gap between rows */
            margin-top: 10px;
            width: 100%;
        }
        
        .gallery-row {
            display: flex;
            gap: 2px; /* 2px gap between photos in a row */
            width: 100%;
        }
        
        .photo-item {
            position: relative;
            cursor: pointer;
            transition: filter 0.2s, box-shadow 0.2s;
            flex-shrink: 0;
            margin: 1px; /* 1px margin around each photo for additional spacing */
        }
        
        .photo-item:hover {
            filter: brightness(0.9);
            box-shadow: 0 8px 16px rgba(0,0,0,0.3);
        }
        
        .photo-item img {
            width: 100%;
            height: 100%;
            object-fit: cover;
            display: block;
            border-radius: 4px; /* Slight rounding for better appearance */
        }
        
        /* Photo viewer */
        .photo-viewer { 
            position: fixed; 
            top: 0; 
            left: 0; 
            width: 100%; 
            height: 100%; 
            background-color: rgba(0,0,0,0.9); 
            display: none; 
            justify-content: center; 
            align-items: center; 
            z-index: 1000; 
        }
        
        .photo-viewer img { 
            max-width: 90%; 
            max-height: 90%; 
            object-fit: contain; 
            border-radius: 8px; 
            box-shadow: 0 8px 32px rgba(0,0,0,0.5); 
        }
        
        .viewer-close { 
            position: absolute; 
            top: 20px; 
            right: 30px; 
            color: white; 
            font-size: 2em; 
            font-weight: bold; 
            cursor: pointer; 
            background-color: rgba(0,0,0,0.7); 
            padding: 10px 15px; 
            border-radius: 50%; 
            transition: background-color 0.3s; 
        }
        
        .viewer-close:hover { 
            background-color: rgba(0,0,0,0.9); 
        }
        
        .viewer-nav {
            position: absolute;
            top: 50%;
            transform: translateY(-50%);
            background: rgba(0,0,0,0.7);
            color: white;
            border: none;
            font-size: 2em;
            padding: 10px 20px;
            cursor: pointer;
            border-radius: 5px;
            transition: background-color 0.3s;
        }
        
        .viewer-nav:hover { 
            background: rgba(0,0,0,0.9); 
        }
        
        .viewer-prev { 
            left: 20px; 
        }
        
        .viewer-next { 
            right: 20px; 
        }
        
        /* Responsive design with maintained 2px margins */
        @media (max-width: 900px) {
            .container { 
                width: 98%; 
                margin: 10px auto; 
            }
            
            .gallery-area { 
                padding: 0 20px !important; 
            }
            
            .person-info {
                flex-direction: column;
                text-align: center;
                gap: 15px;
            }
            
            .person-stats {
                grid-template-columns: repeat(2, 1fr);
                gap: 10px;
            }
            
            .justified-gallery {
                gap: 2px; /* Maintain 2px gaps on tablet */
            }
            
            .gallery-row {
                gap: 2px; /* Maintain 2px gaps on tablet */
            }
            
            .photo-item {
                margin: 1px; /* Maintain 1px margins on tablet */
            }
        }
        
        @media (max-width: 600px) {
            .gallery-area { 
                padding: 0 10px !important; 
            }
            
            .person-stats {
                grid-template-columns: 1fr;
                gap: 8px;
            }
            
            .person-details h2 {
                font-size: 1.5em;
            }
            
            .person-representative {
                width: 80px;
                height: 80px;
            }
            
            .justified-gallery {
                gap: 2px; /* Maintain 2px gaps on mobile */
            }
            
            .gallery-row {
                gap: 2px; /* Maintain 2px gaps on mobile */
            }
            
            .photo-item {
                margin: 1px; /* Maintain 1px margins on mobile */
            }
            
            .viewer-nav {
                font-size: 1.5em;
                padding: 8px 15px;
            }
            
            .viewer-close {
                font-size: 1.5em;
                padding: 8px 12px;
            }
        }
    </style>
</head>
<body>
    <div id="photoViewer" class="photo-viewer">
        <span class="viewer-close" onclick="closeViewer()">&times;</span>
        <button class="viewer-nav viewer-prev" onclick="navigateViewer(-1)">‹</button>
        <img id="viewerImage" src="">
        <button class="viewer-nav viewer-next" onclick="navigateViewer(1)">›</button>
    </div>
    
    <div class="container">
        <a href="/" class="back-button">← Back to All Persons</a>
        
        <div class="header">
            <h1>📼️ {{ person.name }}</h1>
        </div>
        
        <div class="person-info">
            {% if person.representative_image %}
            <img src="{{ person.representative_image.thumbnail_url }}" alt="{{ person.name }}" class="person-representative">
            {% endif %}
            <div class="person-details">
                <h2>{{ person.name }}</h2>
                <div class="person-stats">
                    <div class="stat-item">
                        <div class="stat-value">{{ person.face_count }}</div>
                        <div class="stat-label">Total Faces</div>
                    </div>
                    <div class="stat-item">
                        <div class="stat-value">{{ person.images|length }}</div>
                        <div class="stat-label">Photos</div>
                    </div>
                    <div class="stat-item">
                        <div class="stat-value">{{ "%.2f"|format(person.avg_quality) }}</div>
                        <div class="stat-label">Avg Quality</div>
                    </div>
                </div>
            </div>
        </div>
    </div>
    
    <div class="gallery-area">
        <h2>📼️ Photos of {{ person.name }} Grouped by Date</h2>
        <div id="person-gallery">
            <!-- Date-grouped galleries will be generated by JavaScript -->
        </div>
    </div>
    
    <script>
        let currentPersonGallery = null;
        let currentImageIndex = 0;
        
        // Person data from server
        const personData = {{ person_gallery_json | safe }};
        
        // Helper function to format dates
        function formatDate(timestamp) {
            const date = new Date(timestamp * 1000);
            const today = new Date();
            const yesterday = new Date(today.getTime() - 24 * 60 * 60 * 1000);
            
            // Check if it's today
            if (date.toDateString() === today.toDateString()) {
                return 'Today';
            }
            
            // Check if it's yesterday
            if (date.toDateString() === yesterday.toDateString()) {
                return 'Yesterday';
            }
            
            // Check if it's this year
            if (date.getFullYear() === today.getFullYear()) {
                return date.toLocaleDateString('en-US', { 
                    month: 'long', 
                    day: 'numeric' 
                });
            }
            
            // Full date for other years
            return date.toLocaleDateString('en-US', { 
                year: 'numeric',
                month: 'long', 
                day: 'numeric' 
            });
        }
        
        // Group images by date
        function groupImagesByDate(images) {
            const groups = {};
            
            images.forEach((img, index) => {
                const date = new Date(img.file_modified_time * 1000);
                const dateKey = date.toDateString();
                
                if (!groups[dateKey]) {
                    groups[dateKey] = {
                        timestamp: img.file_modified_time,
                        images: []
                    };
                }
                
                // Add original index for viewer navigation
                groups[dateKey].images.push({
                    ...img,
                    originalIndex: index
                });
            });
            
            // Sort groups by date (newest first)
            const sortedGroups = Object.entries(groups)
                .sort(([,a], [,b]) => b.timestamp - a.timestamp)
                .map(([dateKey, group]) => ({
                    dateKey,
                    formattedDate: formatDate(group.timestamp),
                    timestamp: group.timestamp,
                    images: group.images
                }));
            
            return sortedGroups;
        }
        
        // Perfect Google Photos Justified Gallery Algorithm with 2px gaps
        function createJustifiedGallery(container, images, targetRowHeight = 200, maxRowHeight = 300) {
            if (!images || images.length === 0) return;
            
            const containerWidth = container.offsetWidth - 4; // Account for padding and margins
            let currentRow = [];
            let currentRowWidth = 0;
            const gap = 2; // 2px gap between images
            const photoMargin = 2; // 2px total margin per photo (1px each side)
            
            // Pre-calculate aspect ratios for all images
            const imagesWithAspectRatio = images.map(img => {
                const aspectRatio = img.thumbnail_width && img.thumbnail_height 
                    ? img.thumbnail_width / img.thumbnail_height 
                    : 1.0;
                return { ...img, aspectRatio };
            });
            
            function finalizeRow(isLastRow = false) {
                if (currentRow.length === 0) return;
                
                let rowHeight = targetRowHeight;
                
                if (!isLastRow && currentRow.length > 1) {
                    // Calculate optimal row height for perfect justification
                    const totalAspectRatio = currentRow.reduce((sum, img) => sum + img.aspectRatio, 0);
                    const totalGapWidth = (currentRow.length - 1) * gap;
                    const totalMarginWidth = currentRow.length * photoMargin;
                    const availableWidth = containerWidth - totalGapWidth - totalMarginWidth;
                    
                    // Calculate row height that will make total width exactly match container
                    rowHeight = availableWidth / totalAspectRatio;
                    rowHeight = Math.min(maxRowHeight, Math.max(120, rowHeight));
                }
                
                // Create row element
                const rowElement = document.createElement('div');
                rowElement.className = 'gallery-row';
                
                // Calculate actual widths and ensure perfect fit
                let totalCalculatedWidth = 0;
                const photoItems = [];
                
                // First pass: calculate ideal widths
                currentRow.forEach((img, index) => {
                    let width = Math.floor(rowHeight * img.aspectRatio);
                    totalCalculatedWidth += width;
                    photoItems.push({ img, width, height: rowHeight });
                });
                
                // Second pass: adjust for perfect justification (if not last row)
                if (!isLastRow && currentRow.length > 1) {
                    const totalGapWidth = (currentRow.length - 1) * gap;
                    const totalMarginWidth = currentRow.length * photoMargin;
                    const targetWidth = containerWidth - totalGapWidth - totalMarginWidth;
                    const widthDifference = targetWidth - totalCalculatedWidth;
                    
                    // Distribute the difference across all items
                    let remainingDifference = widthDifference;
                    
                    photoItems.forEach((item, index) => {
                        if (index < photoItems.length - 1) {
                            // Distribute proportionally, but at least 1px adjustment for significant differences
                            const adjustment = remainingDifference > 0 
                                ? Math.max(1, Math.floor(remainingDifference / (photoItems.length - index)))
                                : Math.min(-1, Math.ceil(remainingDifference / (photoItems.length - index)));
                            
                            if (Math.abs(remainingDifference) >= photoItems.length - index) {
                                item.width += adjustment;
                                remainingDifference -= adjustment;
                            }
                        } else {
                            // Last item gets any remaining difference
                            item.width += remainingDifference;
                        }
                    });
                }
                
                // Create DOM elements with calculated dimensions
                photoItems.forEach((item, index) => {
                    const photoItem = document.createElement('div');
                    photoItem.className = 'photo-item';
                    photoItem.style.width = `${item.width}px`;
                    photoItem.style.height = `${item.height}px`;
                    photoItem.onclick = () => openViewer(item.img.originalIndex);
                    
                    const imgElement = document.createElement('img');
                    imgElement.src = item.img.thumbnail_url;
                    imgElement.alt = item.img.filename;
                    imgElement.loading = 'lazy';
                    
                    photoItem.appendChild(imgElement);
                    rowElement.appendChild(photoItem);
                });
                
                container.appendChild(rowElement);
                
                // Reset for next row
                currentRow = [];
                currentRowWidth = 0;
            }
            
            // Process images into rows
            imagesWithAspectRatio.forEach((img, index) => {
                const estimatedWidth = targetRowHeight * img.aspectRatio;
                
                // Calculate total width including gaps and margins
                const gapWidth = currentRow.length > 0 ? gap : 0;
                const projectedRowWidth = currentRowWidth + estimatedWidth + gapWidth + photoMargin;
                
                // Decide whether to add to current row or start new row
                if (projectedRowWidth <= containerWidth || currentRow.length === 0) {
                    // Add to current row
                    currentRow.push(img);
                    currentRowWidth += estimatedWidth + gapWidth + photoMargin;
                } else {
                    // Current row is full, finalize it and start new row
                    finalizeRow();
                    currentRow.push(img);
                    currentRowWidth = estimatedWidth + photoMargin;
                }
            });
            
            // Finalize the last row
            if (currentRow.length > 0) {
                finalizeRow(true);
            }
        }
        
        // Function to adjust date group widths for shrink-wrapping
        function adjustDateGroupWidths() {
            document.querySelectorAll('.date-group').forEach(dateGroup => {
                const justifiedGallery = dateGroup.querySelector('.justified-gallery');
                const galleryRows = justifiedGallery.querySelectorAll('.gallery-row');
                
                if (galleryRows.length === 1) {
                    // Single row - shrink wrap to content
                    dateGroup.classList.add('single-row');
                    dateGroup.classList.remove('multi-row');
                    
                    // Calculate the actual width needed
                    const row = galleryRows[0];
                    const photoItems = row.querySelectorAll('.photo-item');
                    let totalWidth = 0;
                    
                    photoItems.forEach((item, index) => {
                        totalWidth += item.offsetWidth;
                        totalWidth += 2; // Add margin
                        if (index < photoItems.length - 1) {
                            totalWidth += 2; // Add gap between photos
                        }
                    });
                    
                    // Set the exact width needed
                    justifiedGallery.style.width = `${totalWidth}px`;
                    
                } else {
                    // Multiple rows - use full width
                    dateGroup.classList.add('multi-row');
                    dateGroup.classList.remove('single-row');
                    justifiedGallery.style.width = '100%';
                }
            });
        }
        
        // Initialize date-grouped gallery
        function initializeGallery() {
            const mainContainer = document.getElementById('person-gallery');
            if (mainContainer && personData.images) {
                // Clear existing content
                mainContainer.innerHTML = '';
                
                // Group images by date
                const dateGroups = groupImagesByDate(personData.images);
                
                // Create gallery for each date group
                dateGroups.forEach(dateGroup => {
                    // Create date group container
                    const dateGroupDiv = document.createElement('div');
                    dateGroupDiv.className = 'date-group';
                    
                    // Create date header
                    const dateHeader = document.createElement('h3');
                    dateHeader.className = 'date-header';
                    dateHeader.textContent = `${dateGroup.formattedDate} (${dateGroup.images.length} photos)`;
                    
                    // Create justified gallery container
                    const galleryContainer = document.createElement('div');
                    galleryContainer.className = 'justified-gallery';
                    
                    // Add elements to DOM
                    dateGroupDiv.appendChild(dateHeader);
                    dateGroupDiv.appendChild(galleryContainer);
                    mainContainer.appendChild(dateGroupDiv);
                    
                    // Create justified gallery for this date group
                    createJustifiedGallery(galleryContainer, dateGroup.images);
                });
                
                // Adjust date group widths after all galleries are created
                setTimeout(() => {
                    adjustDateGroupWidths();
                }, 100); // Small delay to ensure DOM is fully rendered
            }
        }
        
        // Photo viewer functionality
        const viewer = document.getElementById('photoViewer');
        const viewerImage = document.getElementById('viewerImage');

        function openViewer(imageIndex) {
            currentPersonGallery = personData;
            currentImageIndex = imageIndex;
            updateViewerImage();
            viewer.style.display = 'flex';
        }
        
        function closeViewer() {
            viewer.style.display = 'none';
        }
        
        function navigateViewer(direction) {
            if (!currentPersonGallery) return;
            
            currentImageIndex += direction;
            if (currentImageIndex < 0) {
                currentImageIndex = currentPersonGallery.images.length - 1;
            } else if (currentImageIndex >= currentPersonGallery.images.length) {
                currentImageIndex = 0;
            }
            updateViewerImage();
        }
        
        function updateViewerImage() {
            if (currentPersonGallery && currentPersonGallery.images[currentImageIndex]) {
                viewerImage.src = currentPersonGallery.images[currentImageIndex].original_url;
            }
        }
        
        // Keyboard navigation
        document.addEventListener('keydown', e => {
            if (viewer.style.display === 'flex') {
                if (e.key === "Escape") closeViewer();
                if (e.key === "ArrowLeft") navigateViewer(-1);
                if (e.key === "ArrowRight") navigateViewer(1);
            }
        });
        
        // Initialize gallery when page loads
        document.addEventListener('DOMContentLoaded', initializeGallery);
        
        // Re-initialize on window resize with debouncing
        let resizeTimeout;
        window.addEventListener('resize', () => {
            clearTimeout(resizeTimeout);
            resizeTimeout = setTimeout(initializeGallery, 250);
        });
    </script>
</body>
</html>"""


# ==============================================================================
# 12. FLASK ROUTES
# ==============================================================================
@app.route('/image_thumbnail/<content_hash>')
def serve_image_thumbnail(content_hash):
    try:
        thumbnail_path = generate_thumbnail_path(content_hash)
        if os.path.exists(thumbnail_path):
            return send_file(thumbnail_path, mimetype='image/jpeg')
        else:
            return "Thumbnail not found", 404
    except Exception as e:
        print(f"Error serving thumbnail: {e}")
        return "Error serving thumbnail", 500

@app.route('/original_image/<content_hash>')
def serve_original_image(content_hash):
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute("SELECT path FROM images WHERE content_hash = ?", (content_hash,))
        result = cursor.fetchone()
        conn.close()
        
        if result and os.path.exists(result['path']):
            return send_file(result['path'])
        else:
            return "Original image not found", 404
    except Exception as e:
        print(f"Error serving original image: {e}")
        return "Error serving image", 500

@app.route('/face_thumbnail/<int:face_id>')
def face_thumbnail(face_id):
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute('''SELECT f.bbox_x1, f.bbox_y1, f.bbox_x2, f.bbox_y2, i.path
            FROM faces f JOIN images i ON f.image_id = i.id WHERE f.id = ?''', (face_id,))
        result = cursor.fetchone()
        conn.close()
        
        if not result:
            return "Face not found", 404
        
        bbox = [result['bbox_x1'], result['bbox_y1'], result['bbox_x2'], result['bbox_y2']]
        thumbnail_data = generate_face_thumbnail_from_bbox(result['path'], bbox)
        
        if thumbnail_data:
            return Response(thumbnail_data, mimetype='image/jpeg')
        else:
            return "Error generating face thumbnail", 500
    except Exception as e:
        print(f"Error serving face thumbnail: {e}")
        return "Internal error", 500

@app.route('/person_face_thumbnail/<int:person_id>')
def serve_person_face_thumbnail(person_id):
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        
        cursor.execute('''SELECT f.bbox_x1, f.bbox_y1, f.bbox_x2, f.bbox_y2, i.path, f.quality_score
            FROM faces f JOIN images i ON f.image_id = i.id JOIN person_faces pf ON f.id = pf.face_id
            WHERE pf.person_id = ? ORDER BY f.quality_score DESC, f.confidence DESC LIMIT 1''', (person_id,))
        result = cursor.fetchone()
        conn.close()
        
        if not result:
            return "Face not found", 404
        
        bbox = [result['bbox_x1'], result['bbox_y1'], result['bbox_x2'], result['bbox_y2']]
        thumbnail_data = generate_square_face_thumbnail_from_bbox(result['path'], bbox)
        
        if thumbnail_data:
            return Response(thumbnail_data, mimetype='image/jpeg')
        else:
            return "Error generating face thumbnail", 500
    except Exception as e:
        print(f"Error serving person face thumbnail: {e}")
        return "Internal error", 500

@app.route('/')
def index():
    global processing_thread
    
    database_stats = get_database_stats()
    persons_summary = get_persons_summary()
    
    image_directory = request.args.get('image_directory', '', type=str)
    min_samples = request.args.get('min_samples', 2, type=int)
    num_threads = request.args.get('num_threads', DEFAULT_THREAD_COUNT, type=int)
    
    model_accuracy = calculate_model_accuracy()
    
    return render_template_string(HOME_PAGE_TEMPLATE,
        database_stats=database_stats, persons_summary=persons_summary,
        model_state=model_state, model_accuracy=model_accuracy,
        min_samples=min_samples, num_threads=num_threads, image_directory=image_directory,
        suggestions=suggestions_cache, progress_data=progress_data)

@app.route('/person/<int:person_id>')
def person_detail(person_id):
    person = get_person_with_gallery(person_id)
    if not person:
        flash('Person not found.', 'error')
        return redirect(url_for('index'))
    
    return render_template_string(PERSON_PAGE_TEMPLATE, 
        person=person, person_gallery_json=json.dumps(person))

@app.route('/progress')
def progress():
    with progress_lock:
        return jsonify(progress_data)

@app.route('/classify', methods=['POST'])
def classify_photos():
    global processing_thread, model_state
    
    if processing_thread and processing_thread.is_alive():
        return jsonify({'status': 'error', 'message': 'A classification job is already running.'}), 409
    
    eps = model_state['eps']
    suggestion_eps = request.form.get('suggestion_eps', type=float)
    min_samples = request.form.get('min_samples', type=int)
    num_threads = request.form.get('num_threads', type=int)
    image_directory = request.form.get('image_directory', '').strip()
    
    if not image_directory or not os.path.isdir(image_directory):
        return jsonify({'status': 'error', 'message': 'The provided path is not a valid directory.'}), 400
    
    if not all([suggestion_eps, min_samples, num_threads]):
         return jsonify({'status': 'error', 'message': 'Missing required form parameters.'}), 400

    if suggestion_eps <= eps:
        message = f'Suggestion threshold ({suggestion_eps:.4f}) must be greater than current EPS ({eps:.4f}).'
        return jsonify({'status': 'error', 'message': message}), 400
    
    if num_threads < 1 or num_threads > 16:
        return jsonify({'status': 'error', 'message': 'Number of threads must be between 1 and 16.'}), 400
    
    model_state['suggestion_eps'] = suggestion_eps
    save_model_state()
    
    processing_thread = threading.Thread(
        target=process_classification_job,
        args=(image_directory, suggestion_eps, min_samples, num_threads))
    processing_thread.daemon = True
    processing_thread.start()
    
    return jsonify({'status': 'success', 'message': f'Classification started with {num_threads} threads.'})

@app.route('/decide', methods=['POST'])
def decide():
    global suggestions_cache, feedback_history, model_state
    
    try:
        state_updated = False
        data = request.get_json()
        suggestion_id = data.get('suggestion_id')
        decision = data.get('decision')
        confidence = data.get('confidence', 1.0)
        timestamp = data.get('timestamp', time.time())
        
        if suggestion_id is None or not (0 <= suggestion_id < len(suggestions_cache)):
            return jsonify({'status': 'error', 'message': 'Invalid suggestion ID'}), 400
        
        suggestion = suggestions_cache[suggestion_id]
        
        if decision == 'yes':
            conn = get_db_connection()
            cursor = conn.cursor()
            cursor.execute('''INSERT INTO person_faces (person_id, face_id, distance_to_centroid)
                VALUES (?, ?, ?)''', (suggestion['person_id'], suggestion['face_id'], suggestion['distance']))
            
            cursor.execute('''UPDATE persons 
                SET face_count = (SELECT COUNT(*) FROM person_faces WHERE person_id = ?),
                    updated_at = CURRENT_TIMESTAMP WHERE id = ?''', 
                (suggestion['person_id'], suggestion['person_id']))
            
            conn.commit()
            conn.close()
        
        if decision in ['yes', 'no']:
            feedback_entry = {
                'distance': suggestion['distance'], 'decision': decision,
                'confidence': confidence, 'timestamp': timestamp
            }
            feedback_history.append(feedback_entry)
            state_updated = True

            model_state['total_decisions'] = model_state.get('total_decisions', 0) + 1
            accuracy = calculate_model_accuracy()
            accuracy_history = model_state.setdefault('accuracy_history', [])
            accuracy_history.append(accuracy)
            if len(accuracy_history) > ACCURACY_HISTORY_MAX:
                model_state['accuracy_history'] = accuracy_history[-ACCURACY_HISTORY_MAX:]

            total_decisions = model_state['total_decisions']
            if (total_decisions > MIN_FEEDBACK_FOR_UPDATE and
                    total_decisions % UPDATE_FREQUENCY == 0):
                recent_feedback = feedback_history[-UPDATE_FREQUENCY:]
                positive_distances = [entry['distance'] for entry in recent_feedback if entry['decision'] == 'yes']
                negative_distances = [entry['distance'] for entry in recent_feedback if entry['decision'] == 'no']

                if positive_distances or negative_distances:
                    current_eps = model_state.get('eps', DEFAULT_MODEL_STATE['eps'])
                    target_eps = current_eps
                    if positive_distances and negative_distances:
                        target_eps = ((sum(positive_distances) / len(positive_distances)) +
                                      (sum(negative_distances) / len(negative_distances))) / 2
                    elif positive_distances:
                        target_eps = sum(positive_distances) / len(positive_distances)
                    else:
                        target_eps = sum(negative_distances) / len(negative_distances)

                    updated_eps = current_eps + LEARNING_RATE * (target_eps - current_eps)
                    model_state['eps'] = max(EPS_MIN_BOUND, min(EPS_MAX_BOUND, updated_eps))

        if state_updated:
            save_model_state()

        return jsonify({'status': 'success'})
        
    except Exception as e:
        print(f"Error in decide route: {e}")
        return jsonify({'status': 'error', 'message': str(e)}), 500

@app.route('/reset_model', methods=['POST'])
def reset_model():
    global model_state, feedback_history
    
    model_state = DEFAULT_MODEL_STATE.copy()
    feedback_history = []
    save_model_state()
    
    flash('Model parameters have been reset to defaults.', 'info')
    return redirect(url_for('index'))

@app.route('/clear', methods=['POST'])
def clear_data():
    global suggestions_cache, feedback_history, model_state, processing_thread
    
    if processing_thread and processing_thread.is_alive():
        flash('Cannot clear data while processing is active. Please wait for completion.', 'error')
        return redirect(url_for('index'))
    
    try:
        conn = get_db_connection()
        cursor = conn.cursor()
        
        cursor.execute("DELETE FROM person_faces")
        cursor.execute("DELETE FROM persons")
        cursor.execute("DELETE FROM faces")
        cursor.execute("DELETE FROM images")
        cursor.execute("DELETE FROM app_settings")
        
        conn.commit()
        conn.close()
        
        suggestions_cache = []
        feedback_history = []
        model_state = DEFAULT_MODEL_STATE.copy()
        reset_progress()
        
        if os.path.exists(app.config['THUMBNAILS_DIR']):
            shutil.rmtree(app.config['THUMBNAILS_DIR'])
        if os.path.exists(app.config['SUGGESTION_CROPS_DIR']):
            shutil.rmtree(app.config['SUGGESTION_CROPS_DIR'])
            
        os.makedirs(app.config['THUMBNAILS_DIR'], exist_ok=True)
        os.makedirs(app.config['SUGGESTION_CROPS_DIR'], exist_ok=True)
        
        flash('All database data and thumbnails have been completely cleared.', 'info')
        
    except Exception as e:
        flash(f'Error clearing database: {str(e)}', 'error')
    
    return redirect(url_for('index'))

# ==============================================================================
# 13. APPLICATION STARTUP
# ==============================================================================
if __name__ == '__main__':
    print(f"📼️ Starting Face Clustering with Square Thumbnails...")
    print(f"🎯 Current model EPS: {model_state['eps']:.4f}")
    print(f"🎯 Total decisions so far: {model_state['total_decisions']}")
    print(f"🎖️ Model accuracy: {calculate_model_accuracy():.1%}")
    print(f"🧵 Default thread count: {DEFAULT_THREAD_COUNT}")
    print(f"🗃️ Database file: {app.config['DATABASE']}")
    print(f"📼️ Thumbnails directory: {app.config['THUMBNAILS_DIR']}")
    print(f"📐 Thumbnail max dimensions: {THUMBNAIL_MAX_WIDTH}x{THUMBNAIL_MAX_HEIGHT}")
    print(f"🎨 Thumbnail quality: {THUMBNAIL_QUALITY}%")
    print(f"🔄 EXIF orientation correction: ENABLED")
    print(f"🔲 Square face thumbnails with overlaid labels: ENABLED")
    
    db_stats = get_database_stats()
    print(f"🖼️ Total images in database: {db_stats['total_images']}")
    print(f"👤 Total faces in database: {db_stats['total_faces']}")
    print(f"👥️ Total persons identified: {db_stats['total_persons']}")
    print(f"📼️ Total orientation-corrected thumbnails: {db_stats['total_thumbnails']}")
    
    app.run(host='0.0.0.0', port=5000, debug=False, threaded=True)


c:\Users\ishub\anaconda3\envs\torch_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Error loading model state: name 'get_db_connection' is not defined
Initializing FaceAnalysis model...
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\ishub/.insightface\models\buffalo_l\1k3d68.onnx landmark_3

 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.14:5000
Press CTRL+C to quit
127.0.0.1 - - [07/Sep/2025 13:23:13] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:19] "POST /classify HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:20] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:20] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:21] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:21] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:22] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:22] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:23] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:23] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:24] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:24] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:23:25] "GET /progress HTTP/

Error fixing orientation for D:\python\photos\01\DSC_0369.JPG: cannot identify image file 'D:\\python\\photos\\01\\DSC_0369.JPG'Error fixing orientation for D:\python\photos\01\DSC_0370.JPG: cannot identify image file 'D:\\python\\photos\\01\\DSC_0370.JPG'

Error fixing orientation for D:\python\photos\01\DSC_0368.JPG: cannot identify image file 'D:\\python\\photos\\01\\DSC_0368.JPG'
Error fixing orientation for D:\python\photos\01\DSC_0371.JPG: cannot identify image file 'D:\\python\\photos\\01\\DSC_0371.JPG'
Error fixing orientation for D:\python\photos\01\DSC_0373.JPG: cannot identify image file 'D:\\python\\photos\\01\\DSC_0373.JPG'
Error fixing orientation for D:\python\photos\01\DSC_0372.JPG: cannot identify image file 'D:\\python\\photos\\01\\DSC_0372.JPG'
Error fixing orientation for D:\python\photos\01\DSC_0374.JPG: cannot identify image file 'D:\\python\\photos\\01\\DSC_0374.JPG'
Error fixing orientation for D:\python\photos\01\DSC_0376.JPG: cannot identify image file 'D:\\py

127.0.0.1 - - [07/Sep/2025 13:25:07] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:08] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:08] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:09] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:09] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:10] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:10] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:11] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:11] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:12] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:12] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:13] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:13] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:14] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:25:14] "GET /progr

...

127.0.0.1 - - [07/Sep/2025 13:26:45] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:45] "GET /progress HTTP/1.1" 200 -


Error adding image to database: UNIQUE constraint failed: images.content_hash
Error adding image to database: UNIQUE constraint failed: images.content_hash
Error adding image to database: UNIQUE constraint failed: images.content_hash
Error adding image to database: UNIQUE constraint failed: images.content_hash
Error adding image to database: UNIQUE constraint failed: images.content_hash
Error adding image to database: UNIQUE constraint failed: images.content_hash


127.0.0.1 - - [07/Sep/2025 13:26:46] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:46] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:47] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:47] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:48] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:48] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:49] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:49] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:50] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:50] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:51] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:51] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:52] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:52] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:26:53] "GET /progr

Error adding image to database: UNIQUE constraint failed: images.content_hash


127.0.0.1 - - [07/Sep/2025 13:27:04] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:04] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:05] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:05] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:06] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:06] "GET /progress HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:08] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:08] "GET /face_thumbnail/7740 HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:08] "GET /face_thumbnail/5148 HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:08] "GET /face_thumbnail/5306 HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:08] "GET /face_thumbnail/5163 HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:08] "GET /face_thumbnail/4734 HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:08] "GET /face_thumbnail/4716 HTTP/1.1" 200 -
127.0.0.1 - - [07/Sep/2025 13:27:08] "GET /face_thumbnail/